# Module 01 — What Is PyTorch, and Your First Tensors

**Prerequisites:** Module 00 (Prerequisites & Python Refresher)
**Time:** ~90 minutes

## Learning Objectives

- Explain what PyTorch is and why it exists (in one sentence you could say to a colleague).
- Create tensors using every major factory function.
- Inspect a tensor's `shape`, `dtype`, and `device`, and explain what each one means.
- Cast between dtypes and convert between NumPy arrays and PyTorch tensors.
- Understand memory layout, contiguous tensors, and in-place operations.
- Know the standard shape conventions for images, text, and tabular data.


## Why PyTorch Exists

Suppose you want to train a neural network. You need three things, all at once:

1. **Fast numerical computation** on large arrays of numbers (millions to billions of them).
2. **Automatic differentiation** — a way to compute gradients (the ingredient gradient descent needs) without you hand-deriving calculus for every model.
3. **GPU support**, because a modern GPU can perform the same arithmetic 50-100x faster than a CPU for the kind of embarrassingly parallel math neural networks require.

NumPy gives you (1). Nothing about NumPy gives you (2) or (3) — a NumPy array has no idea it's part of a larger computation, and it cannot run on a GPU.

PyTorch's core contribution is a single object, `torch.Tensor`, that behaves like a NumPy array but *additionally* knows how to:

- live on a GPU,
- track the sequence of operations applied to it, so gradients can be computed automatically later.

That's it. That one idea — an array that can live on a GPU and remembers its own history — is what everything else in this course is built on top of.

PyTorch was released by Meta (then Facebook) AI Research in 2017. It won out over earlier frameworks in large part because it runs your code *immediately*, line by line ("eager execution"), rather than requiring you to build an abstract computation graph before running anything. That means ordinary Python debugging tools — `print()`, breakpoints, `if`/`else` — work exactly as you'd expect.


## Setting Up

Run the cell below. If PyTorch isn't installed, uncomment the `pip install` line.


In [ ]:
# !pip install torch --quiet

import torch
import numpy as np
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())


## What Is a Tensor?

A **tensor** is a container for numbers, arranged along zero or more dimensions. That's the whole definition — the rest is vocabulary for describing *how* the numbers are arranged.

| Name | Dimensions | Example |
|---|---|---|
| Scalar | 0 | a single number, `7.0` |
| Vector | 1 | a list of numbers, `[1, 2, 3]` |
| Matrix | 2 | a grid of numbers (rows × columns) |
| Tensor (general) | 3+ | a stack of matrices, a batch of images, etc. |

In casual PyTorch usage, everyone just says "tensor" regardless of how many dimensions it has — a scalar and an image batch are both "tensors," just with different shapes.

Let's create a few.


In [ ]:
# From a plain Python list
x = torch.tensor([1, 2, 3])
print(x)

# A 2D tensor (matrix) from a nested list
y = torch.tensor([[1, 2], [3, 4]])
print(y)

# Tensors filled with zeros / ones of a given shape
z = torch.zeros(2, 3)
w = torch.ones(3)
print(z)
print(w)

# Random values, useful for quickly testing shapes before you have real data
r = torch.randn(2, 3)   # randn = random values from a standard normal distribution
print(r)


### Why not just use a Python list?

You *could* store `[1, 2, 3]` as a plain list. Here's what you'd lose:

- **No vectorized math.** `[1, 2, 3] * 2` on a Python list gives `[1, 2, 3, 1, 2, 3]` (list repetition!), not element-wise multiplication. You'd need a manual loop.
- **No shape/dtype/device metadata.** A list doesn't know it represents a "3x224x224 RGB image" — it's just numbers.
- **No GPU support.** Lists live in regular Python memory and can't be transferred to a GPU.
- **No autograd.** Lists can't record the operations applied to them for later gradient computation.

Tensors solve all four.


## Complete Tensor Factory Tour

PyTorch provides many ways to create tensors. Here's the full reference — you don't need to memorize them all, but you should recognize each one when you see it.


In [ ]:
# --- Creation from data ---
from_list = torch.tensor([1.0, 2.0, 3.0])          # from Python list
print("from list:", from_list, "| shape:", from_list.shape)

# --- Filled with specific values ---
zeros = torch.zeros(3, 4)                            # all zeros
ones = torch.ones(2, 5)                              # all ones
full = torch.full((2, 3), fill_value=7.0)            # all 7s
empty = torch.empty(2, 3)                            # uninitialized (garbage values — fast but dangerous)

print("zeros:", zeros.shape)
print("ones:", ones.shape)
print("full:", full)
print("empty (uninitialized, values are garbage):", empty)


In [ ]:
# --- Sequences ---
arange = torch.arange(0, 10, 2)                     # [0, 2, 4, 6, 8] — like Python range()
linspace = torch.linspace(0, 1, 5)                   # 5 evenly-spaced values from 0 to 1
print("arange:", arange)
print("linspace:", linspace)

# --- Special matrices ---
eye = torch.eye(3)                                   # 3×3 identity matrix
print("eye:\n", eye)

# --- Random ---
rand = torch.rand(2, 3)                              # uniform random in [0, 1)
randn = torch.randn(2, 3)                            # normal distribution (mean=0, std=1)
randint = torch.randint(0, 10, (2, 3))               # random integers in [0, 10)
print("rand (uniform [0,1)):", rand)
print("randn (normal):", randn)
print("randint [0,10):", randint)


In [ ]:
# --- "_like" variants: create a tensor with the same shape/dtype as another ---
template = torch.randn(4, 5)
zeros_like = torch.zeros_like(template)    # same shape (4,5) and dtype, filled with zeros
ones_like = torch.ones_like(template)
rand_like = torch.rand_like(template)

print("template shape:", template.shape)
print("zeros_like shape:", zeros_like.shape)
print("randn_like dtype:", rand_like.dtype)  # inherits float32 from template


**When to use which:**

| Function | Use case |
|---|---|
| `torch.tensor(data)` | You have specific values to store |
| `torch.zeros/ones/full` | Initialize a tensor of known shape |
| `torch.randn` | Quick test data, weight initialization |
| `torch.arange/linspace` | Create sequences (feature grids, time steps) |
| `torch.eye` | Identity matrix (e.g., for one-hot encoding) |
| `torch.*_like(template)` | Match another tensor's shape without computing its shape manually |
| `torch.empty` | Performance-critical code where you'll immediately overwrite all values |


## The Three Properties Every Tensor Has

Every tensor has exactly three things worth checking whenever something goes wrong: **shape**, **dtype**, and **device**.


In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

print("shape :", x.shape)    # the size along each dimension
print("dtype :", x.dtype)    # what kind of number is stored (float32, int64, ...)
print("device:", x.device)   # where the data physically lives (cpu or cuda)
print("ndim  :", x.ndim)     # number of dimensions (= len(shape))
print("numel :", x.numel())  # total number of elements


**`shape`** tells you the size along each dimension. `torch.Size([2, 3])` means "2 rows, 3 columns" — or more generally, "dimension 0 has size 2, dimension 1 has size 3." You will check `.shape` constantly; it's the #1 debugging tool in PyTorch.

**`dtype`** tells you the numeric type. The most common ones:

| dtype | Meaning | Typical use |
|---|---|---|
| `torch.float32` | 32-bit decimal number | Default for model weights and inputs |
| `torch.float64` | 64-bit decimal (double) | NumPy default, rarely needed in PyTorch |
| `torch.float16` | 16-bit decimal (half) | Mixed-precision training on GPU |
| `torch.int64` (`torch.long`) | 64-bit whole number | Class labels for classification |
| `torch.int32` | 32-bit whole number | Less common |
| `torch.bool` | True/False | Masks, conditions |

**Rule to remember now, understand later:** mixing `float32` model weights with `int64` labels in the wrong place is a common source of errors. `nn.CrossEntropyLoss`, for instance, expects predictions as `float` and labels as `long` — we'll hit this directly in a later module.

**`device`** tells you whether the tensor lives in ordinary system memory (`cpu`) or GPU memory (`cuda`). Operations between two tensors require them to be on the *same* device — this is the single most common beginner error.


## dtype Casting

You'll frequently need to convert between dtypes. There are two ways: shorthand methods and the general `.to()` method.


In [ ]:
x = torch.tensor([1, 2, 3])   # defaults to int64
print("original dtype:", x.dtype)

# Shorthand methods
x_float = x.float()           # int64 -> float32
x_double = x.double()         # int64 -> float64
x_long = x_float.long()       # float32 -> int64 (truncates decimals!)
x_bool = x.bool()             # nonzero -> True, zero -> False

print("float:", x_float.dtype)
print("double:", x_double.dtype)
print("long:", x_long.dtype)
print("bool:", x_bool)

# General .to() method — works for any dtype
x_half = x.to(torch.float16)
print("half:", x_half.dtype)


### 🐛 Debugging Challenge: dtype Mismatch

Many PyTorch functions are strict about dtypes. Here's a common error:


In [ ]:
# This is a VERY common error when computing loss
predictions = torch.tensor([0.9, 0.1, 0.8])   # float32 (correct)
labels = torch.tensor([1, 0, 1])                # int64 (correct for CrossEntropyLoss, WRONG for MSELoss)

try:
    # MSELoss expects both tensors to be the same dtype
    import torch.nn as nn
    loss = nn.MSELoss()(predictions, labels)
except RuntimeError as e:
    print("Error:", e)

# Fix: cast labels to float
loss = nn.MSELoss()(predictions, labels.float())
print("Fixed MSE loss:", loss.item())


## NumPy ↔ PyTorch Interop

Since much of the Python data science ecosystem uses NumPy, you'll frequently need to convert between the two. **Important:** by default, the conversion shares memory — modifying one modifies the other.


In [ ]:
# NumPy -> PyTorch
np_array = np.array([1.0, 2.0, 3.0])
tensor_from_numpy = torch.from_numpy(np_array)
print("from numpy:", tensor_from_numpy)

# PyTorch -> NumPy
back_to_numpy = tensor_from_numpy.numpy()
print("back to numpy:", back_to_numpy)
print("type:", type(back_to_numpy))


In [ ]:
# ⚠️ WARNING: shared memory — modifying one changes the other!
np_arr = np.array([10.0, 20.0, 30.0])
tensor = torch.from_numpy(np_arr)

print("Before:", tensor)
np_arr[0] = 999.0           # modify the NumPy array
print("After modifying NumPy:", tensor)   # tensor changed too!

# To break the link, use .clone()
np_arr2 = np.array([1.0, 2.0, 3.0])
independent_tensor = torch.from_numpy(np_arr2).clone()  # independent copy
np_arr2[0] = 999.0
print("Independent tensor (unchanged):", independent_tensor)


**Key rule:** `torch.from_numpy()` and `.numpy()` share memory. If you want an independent copy, call `.clone()` on the tensor (or `.copy()` on the numpy array) immediately after conversion.

Also note: `.numpy()` only works on CPU tensors. If your tensor is on a GPU, you need `.cpu().numpy()` — you'll meet this pattern in the GPU training notebook.


### 🔮 Predict before you run

Before running the next cell, predict: what will `x.shape` be for `torch.tensor([[1, 2, 3, 4]])` — note the *extra* pair of brackets around the whole thing?


In [ ]:
x = torch.tensor([[1, 2, 3, 4]])
print(x.shape)


If you predicted `torch.Size([1, 4])` — one row, four columns — you're already reading shapes correctly. The outer brackets create dimension 0 (size 1), and the inner list creates dimension 1 (size 4). This distinction between `[1, 2, 3, 4]` (shape `(4,)`) and `[[1, 2, 3, 4]]` (shape `(1, 4)`) trips up nearly everyone at first, and matters a lot once you start feeding batches of data into models.


## 🐛 Debugging Challenge: The Device Mismatch

This is, by a wide margin, the error you will see most often as a beginner. Let's produce it on purpose so you recognize it instantly later.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

a = torch.randn(3, 3)              # created on CPU by default
b = torch.randn(3, 3).to(device)   # explicitly moved to `device`

try:
    result = a + b
except RuntimeError as e:
    print("RuntimeError caught:")
    print(e)


If you're running this on a machine *without* a GPU, `device` is `"cpu"` for both tensors, so no error occurs — that's expected and fine; the error only appears on a GPU-enabled machine. Either way, the fix is the same: **move every tensor involved in an operation onto the same device before computing**, typically by calling `.to(device)` right after creating or loading each tensor.

```python
a = a.to(device)
b = b.to(device)
result = a + b   # now safe
```


## Reshaping: Same Data, Different Shape

Reshaping doesn't move or copy the underlying numbers — it just changes how they're *grouped* into dimensions. Think of it as relabeling, not rearranging.


In [ ]:
x = torch.arange(12)   # tensor([0, 1, 2, ..., 11]), shape (12,)
print("original:", x.shape)

y = x.reshape(3, 4)     # same 12 numbers, viewed as 3 rows of 4
print("reshaped:", y.shape)
print(y)

z = x.reshape(2, 2, 3)  # same 12 numbers, viewed as 2 blocks of 2x3
print("reshaped again:", z.shape)


`reshape` requires the total number of elements to stay the same — 12 elements can become `(3, 4)`, `(2, 2, 3)`, or `(12,)`, but never `(3, 5)` (15 slots for 12 elements). Try it below and see the error PyTorch gives you.


In [ ]:
try:
    bad = x.reshape(3, 5)
except RuntimeError as e:
    print(e)


In [ ]:
# The -1 trick: let PyTorch figure out one dimension automatically
x = torch.arange(12)

print(x.reshape(3, -1))     # -1 becomes 4 (12/3=4)
print(x.reshape(-1, 6))     # -1 becomes 2 (12/6=2)
print(x.reshape(2, 2, -1))  # -1 becomes 3 (12/(2*2)=3)


### `view()` vs `reshape()`

- `.view()` — returns a new tensor sharing the **same memory** as the original. Fails if the tensor isn't contiguous in memory.
- `.reshape()` — returns a view if possible, otherwise copies the data. Always works.

**Recommendation:** use `.reshape()` unless you specifically need to guarantee no copy happens (rare in practice).


### Memory layout and `.contiguous()`

Tensors are stored as a flat block of numbers in memory. "Contiguous" means the elements are laid out in the expected row-major order. Some operations (like `.transpose()`) return a tensor that *views* the same memory in a different order — fast, but not contiguous.


In [ ]:
x = torch.arange(6).reshape(2, 3)
print("x contiguous?", x.is_contiguous())   # True

y = x.T   # transpose — shares memory, but now not contiguous
print("x.T contiguous?", y.is_contiguous())  # False

# .view() would fail on y, but .reshape() works fine
try:
    y.view(6)
except RuntimeError as e:
    print("view() error:", e)

# Fix: make it contiguous first, or just use reshape
print("reshape works:", y.reshape(6))
print("contiguous + view works:", y.contiguous().view(6))


## Squeeze and Unsqueeze

You'll see `x.unsqueeze(dim)` (insert a size-1 dimension) and `x.squeeze()` (remove size-1 dimensions) constantly — these come up when a function expects a "batch" dimension that a single example doesn't naturally have.


In [ ]:
single_image = torch.randn(28, 28)          # one 28x28 grayscale image
print("before:", single_image.shape)

batched = single_image.unsqueeze(0)          # add a batch dimension at position 0
print("after unsqueeze(0):", batched.shape)  # now looks like "a batch of 1 image"

back = batched.squeeze(0)                    # remove it again
print("after squeeze(0):", back.shape)


This matters because PyTorch models almost always expect a **batch dimension**, even if you're only passing in one example. `unsqueeze(0)` is how you turn "one image" into "a batch containing one image."


## In-place Operations

Any PyTorch operation with a trailing underscore modifies the tensor **in place** rather than returning a new one. These are faster (no copy), but can break autograd.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
print("before:", x)

x.add_(10)      # in-place: modifies x directly
print("after add_(10):", x)

x.mul_(2)       # in-place multiply
print("after mul_(2):", x)

x.zero_()       # in-place fill with zeros (you'll see this in gradient zeroing)
print("after zero_():", x)


**Convention:** `operation()` returns a new tensor; `operation_()` modifies in place.

**Warning:** in-place operations on tensors that require gradients can break autograd's computation graph. The main place you'll safely use in-place ops is `grad.zero_()` (clearing gradients) and parameter updates inside `torch.no_grad()` blocks. We'll revisit this in the autograd notebook.


## Real-World Shape Conventions

Different types of data follow standard shape conventions in PyTorch:

| Data type | Convention | Example shape |
|---|---|---|
| Tabular / features | `(batch, features)` | `(64, 10)` = 64 samples, 10 features each |
| Grayscale images | `(batch, channels=1, height, width)` | `(32, 1, 28, 28)` = 32 MNIST images |
| Color images | `(batch, channels=3, height, width)` | `(16, 3, 224, 224)` = 16 ImageNet images |
| Text / sequences | `(batch, sequence_length)` | `(8, 512)` = 8 sentences, each 512 tokens |
| Text embeddings | `(batch, sequence_length, embedding_dim)` | `(8, 512, 768)` |

**Note the channel-first convention:** PyTorch puts channels *before* height and width: `(B, C, H, W)`. This differs from some other frameworks (TensorFlow defaults to `(B, H, W, C)`). Getting this wrong produces shape mismatches in convolutional layers.


In [ ]:
# Simulate real data shapes
# A batch of 8 RGB images, each 32x32 pixels
image_batch = torch.randn(8, 3, 32, 32)
print(f"Image batch: {image_batch.shape}")
print(f"  batch_size = {image_batch.shape[0]}")
print(f"  channels   = {image_batch.shape[1]}")
print(f"  height     = {image_batch.shape[2]}")
print(f"  width      = {image_batch.shape[3]}")

# A batch of tabular data: 64 samples, 13 features
tabular_batch = torch.randn(64, 13)
print(f"\nTabular batch: {tabular_batch.shape}")

# A batch of text embeddings: 4 sentences, each 128 tokens, 256-dim embedding
text_batch = torch.randn(4, 128, 256)
print(f"Text batch: {text_batch.shape}")


## Reproducibility: `torch.manual_seed`

Random tensor creation functions (`randn`, `rand`, `randint`) use a random number generator. To get the **same random values every time** (critical for reproducible experiments), set the seed.


In [ ]:
torch.manual_seed(42)
a = torch.randn(3)

torch.manual_seed(42)  # reset seed to same value
b = torch.randn(3)

print("a:", a)
print("b:", b)
print("identical?", torch.equal(a, b))   # True — same seed produces same numbers


## Exercises

🟢 **Beginner 1:** Create a 1D tensor of the numbers 0 through 9 using `torch.arange(10)`. Print its shape, dtype, and device.

🟢 **Beginner 2:** Create 5 evenly-spaced values between 0 and 1 using `torch.linspace`. Print the result.

🟡 **Intermediate 1:** Reshape that `arange(10)` tensor into shape `(2, 5)`, then into `(5, 2)`. Before running each reshape, predict what the printed tensor will look like.

🟡 **Intermediate 2:** Create a NumPy array `[1, 2, 3]`, convert it to a PyTorch tensor, modify the NumPy array, and verify the tensor changed too. Then do it again with `.clone()` and verify they're independent.

🟡 **Intermediate 3:** Create a tensor of dtype `int64`. Cast it to `float32`, then to `bool`. Print each result and explain the bool conversion.

🔴 **Challenge 1:** Create a tensor of shape `(3, 4)` using `torch.randn`. Without using `.shape`, write code that computes the total number of elements using only `.size()` and indexing.

🔴 **Challenge 2:** Create a mock batch of 4 grayscale images of size 28×28 (shape `(4, 1, 28, 28)`). Extract the first image and squeeze it to shape `(28, 28)`. Then create a mock batch of 8 RGB 64×64 images and verify the shape.


In [ ]:
# Space for your exercise solutions



## Common Mistakes

- **Confusing `torch.Tensor([2,3])` (values 2 and 3) with `torch.zeros(2,3)` (a 2x3 grid of zeros).** Passing data vs. passing a shape are very different calls — watch the function name.
- **Forgetting a tensor needs a batch dimension** before passing it to a model — leads to a shape-mismatch error deep inside the model rather than an obvious one.
- **Assuming reshape can create or destroy elements.** It can't — the total element count must match before and after.
- **Comparing/combining tensors on different devices.** Always `.to(device)` before combining.
- **Forgetting NumPy ↔ Tensor conversion shares memory.** Use `.clone()` if you need independence.
- **Using `torch.empty()` and expecting zeros.** `empty` is uninitialized — values are garbage. Use `torch.zeros()` if you need zeros.
- **Mixing up `.view()` and `.reshape()`** — use `.reshape()` unless you have a specific reason to need `.view()`.

## Mental Model

A tensor is "a box of numbers plus a label describing the box's shape, its number type, and which machine (CPU/GPU) it's sitting on." Whenever something breaks, check the label first — `.shape`, `.dtype`, `.device` — before suspecting the math.

## Key Takeaways

- PyTorch exists to combine array computation + automatic differentiation + GPU support in one object: the tensor.
- Every tensor has a shape, dtype, and device — check all three when debugging.
- Use the right factory function: `zeros`, `ones`, `randn`, `arange`, `linspace`, `eye`, `full`, `empty`.
- dtype casting: `.float()`, `.long()`, `.to(dtype)` — mixing dtypes causes common errors.
- NumPy interop: `torch.from_numpy()` and `.numpy()` share memory; use `.clone()` to break the link.
- Reshaping relabels data; it never creates, destroys, or moves individual values.
- Operations require matching devices; this is the single most common beginner error.
- Know the standard shape conventions: `(B, features)` for tabular, `(B, C, H, W)` for images.

## What's Next

**Module 02 — Tensor Operations, Indexing & Broadcasting** builds directly on this: you'll learn how tensors of *different* shapes can still be combined via broadcasting, and how to slice into specific parts of a tensor.

## Checklist

- [ ] I can create tensors from Python lists and using `torch.zeros`/`torch.ones`/`torch.randn`/`torch.arange`/`torch.linspace`
- [ ] I can explain what shape, dtype, and device each mean
- [ ] I can cast between dtypes using `.float()`, `.long()`, `.to()`
- [ ] I can convert between NumPy arrays and PyTorch tensors, and I know about shared memory
- [ ] I can reshape a tensor and predict whether a given reshape is valid
- [ ] I understand why operations require tensors to share the same device
- [ ] I know the standard shape conventions for images `(B, C, H, W)` and tabular data `(B, features)`
